In [ ]:
%load_ext autoreload
%autoreload 2

import mne
import json
from pathlib import Path

edf_path = Path("data/real_data/test_from_20s.edf")

raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
raw.info

In [ ]:
from ARMBR import run_armbr

frontal_channels = ['EEG FP1-A1', 'EEG FP2-A2', 'EEG FPZ-A1', 'EEG FZ-A2']

eeg_data = raw.get_data(picks='eeg').T  # Transpose to (samples, channels)

# Get indices of frontal channels in the EEG data
eeg_ch_names = raw.copy().pick('eeg').ch_names
blink_indices = [eeg_ch_names.index(ch) for ch in frontal_channels]

# Run ARMBR
x_clean, best_alpha, ref_mask, blink_comp, blink_pattern, removal_matrix = run_armbr(
    X=eeg_data,
    blink_ch_idx=blink_indices,
    exclude_ch_idx=[],  # No exclusion needed
    sfreq=raw.info['sfreq'],
    alpha= 3 # -1  # Auto-select optimal threshold
)

best_alpha # 4.11 for alpha = -1

In [ ]:
import numpy as np

sfreq = raw.info["sfreq"]
annotations = []

if len(ref_mask) > 0:
    blink_samples = np.where(ref_mask)[0]

    if len(blink_samples) > 0:
        # Group consecutive blinks (gap < 50ms)
        gaps = np.diff(blink_samples)
        blink_boundaries = np.where(gaps > int(0.05 * sfreq))[0]

        start_idx = 0
        for boundary in blink_boundaries:
            end_idx = boundary + 1
            blink_segment = blink_samples[start_idx:end_idx]

            annotations.append(
                {
                    "label": "EOG",
                    "startSec": round(blink_segment[0] / sfreq, 3),
                    "endSec": round(blink_segment[-1] / sfreq, 3),
                }
            )

            start_idx = boundary + 1

        # Last blink
        if start_idx < len(blink_samples):
            blink_segment = blink_samples[start_idx:]
            annotations.append(
                {
                    "label": "EOG",
                    "startSec": round(blink_segment[0] / sfreq, 3),
                    "endSec": round(blink_segment[-1] / sfreq, 3),
                }
            )

print(f"Found {len(annotations)} blink events")
for ann in annotations[:5]:
    print(f"Blink: {ann['startSec']}s - {ann['endSec']}s")

In [ ]:
output_path = edf_path.parent / "tmp_armbr.json"

annot_dict = {
    "name": "armbr",
    "opacity": 0.2,
    "visible": True,
    "events": annotations,
}

with output_path.open("w", encoding="utf-8") as f:
    json.dump(annot_dict, f, indent=2)

In [ ]:
# %matplotlib
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt

mne_annotations = mne.Annotations(
    onset=[ann['startSec'] for ann in annotations],
    duration=[ann['endSec'] - ann['startSec'] for ann in annotations],
    description=[ann['label'] for ann in annotations]
)
raw.set_annotations(mne_annotations)

fig = raw.plot(
    duration=30,  # Show 30 seconds initially
    start=0,
    n_channels=15,
    scalings='auto',
    show=True,
    block=True  # Keeps the plot interactive
)